# BSAN 775 Assignment 2 (Chapters 7 to 10)

**Chapters:** 7 to 10

**Points:** 100

**Due:** Sunday, November 15, 2026, 11:59 pm Central

Prairie Wholesale's head of customer success is preparing the budget for next
year's retention program. Before she commits to it, she needs four answers:

- the effect of the free-shipping pilot beyond order size;
- the factors associated with the discounts customers receive;
- the use of the churn model under next year's costs;
- the quality of the churn list if order dates are missing from the new
  customer export.

In each problem you will apply one chapter's method to a question left open in
that chapter.

## Submission

Submit one notebook, downloaded as `.ipynb`, to the Assignment 2 dropbox in
Blackboard. Before you submit, run **Restart and run all** and confirm that no
error is raised in any cell.

## Grading

| Problem | Points |
|---|---|
| 1. Pilot outcomes beyond order size | 25 |
| 2. Drivers of the discount rate | 25 |
| 3. Churn list under next year's costs | 25 |
| 4. Churn list without order dates | 25 |

Forty of the hundred points are written interpretation. In each written part, a
rule is stated, and you will apply it to your own numbers.

A correct number with no interpretation is given part of the marks. The
interpretation belongs in the cells marked **graded**.

If you used an AI assistant, add a line at the end naming the tool and what it
helped with, per Appendix D.

## Running this notebook

Run the setup cell, then the provided cell, then work down. You will enter your
work in the cells marked `# TODO`. Everything else is provided.

**If you are in Google Colab:** this notebook is opened read-only from GitHub.
Click **Copy to Drive** in the toolbar (or *File > Save a copy in Drive*) before
you edit anything, then work in that copy. Edits to the read-only original are
lost when the tab is closed.

For every step, the function you need and the chapter section in which it is
taught are named. Where a step resembles a lab step, the lab is named. You are
not expected to invent anything.

Where a line is left for you to write, it contains `REPLACE_THIS`. That name is
not defined anywhere. If you run a cell you have not completed, the error
`NameError: name 'REPLACE_THIS' is not defined` is raised, and the line to fill
in is shown in the error message. Delete the placeholder and write your code in
its place. Some placeholders stand for several lines. Those lines are described
in the comment beside the placeholder.

A fixed seed is set for every random step: `np.random.default_rng(0)` in
Problem 1 and `random_state=0` in Problems 3 and 4. With these seeds, your
numbers are the same on every run and the same as everyone else's.

In most steps you either add a column to a table you already have or make a new
table from an old one. Whenever you are unsure of a step's result, print it. Use
`.head()` for a table or a column, and the bare name for a single number. The
cells headed `# --- Look: ... ---` are provided and are not graded. Each one
prints the result of the step before it.

Because several forests are fitted in Problem 4, allow about a minute for its
cells.

In [ ]:
# Setup. Run this cell once per session. It installs the packages this
# assignment needs; on Google Colab it also fetches the course data.
%pip install -q pandas scipy statsmodels scikit-learn
import sys
if "google.colab" in sys.modules:
    !test -d pyba-companion || git clone --quiet --depth 1 https://github.com/murtaza-nasir/pyba-companion.git
    sys.path.insert(0, "pyba-companion")   # makes `import pyba` (DATA_DIR) work

## Provided: the data and the chapter objects

The data and four objects from earlier chapters are rebuilt in the provided
cell:

| Object | Contents | Source |
|---|---|---|
| `orders` | one row per order line | Chapter 3 |
| `snap` | the customer snapshot, one row per customer | Chapter 7 |
| `window` | one row per online **order** placed between March 2 and May 31, 2026, with the customer's `pilot` group | Chapter 7, section "The pilot" |
| `X`, `y` | the 20 feature columns and the `churned` label | Chapter 9, section "The train/test discipline" |

In [ ]:
# --- Provided: run this cell second ---
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier

from pyba import DATA_DIR

orders = pd.read_csv(DATA_DIR / "pw_orders.csv", parse_dates=["order_date"])
snap = pd.read_csv(DATA_DIR / "pw_customer_snapshot.csv")

# Chapter 7: one row per order, then the online orders placed in the pilot window
totals = orders.groupby("order_id").agg(
    total=("line_total", "sum"),
    date=("order_date", "first"),
    channel=("channel", "first"),
    customer_id=("customer_id", "first"),
)
window = totals[
    (totals["channel"] == "online")
    & (totals["date"] >= "2026-03-02")
    & (totals["date"] <= "2026-05-31")
].merge(snap[["customer_id", "pilot"]], on="customer_id").dropna(subset=["pilot"])

# Chapter 9: the feature table and the churn label
X = pd.get_dummies(snap.drop(columns=["customer_id", "pilot", "churned"]),
                   columns=["business_type", "region"])
y = snap["churned"]

print(f"window: {len(window)} online orders in the pilot window")
print(f"snap:   {len(snap)} customers; X has {X.shape[1]} feature columns")

---

# Problem 1: Pilot outcomes beyond order size (25 points)

In Chapter 7, the test covered one outcome of the free-shipping pilot, the mean
order total. Free shipping was offered only on orders over \$200. In this problem you
will test two further outcomes, each measured per customer:

| Outcome | Column | Definition |
|---|---|---|
| qualifying orders | `n_qualifying` | the customer's online orders over \$200 in the pilot window |
| spend | `spend` | the total of the customer's online orders in the pilot window |

For each outcome you will report four numbers: the difference in means, the
Welch p-value, the permutation p-value, and the 95% bootstrap interval.

## Look at one customer's orders first

Run the cell below. Customer C0077 is in the pilot group and placed six online
orders in the window. Three of them are over \$200. In the table you build in
1a, this customer has one row, with `n_qualifying` equal to 3 and `spend` equal
to the sum of the six totals, \$1,143.21.

In [ ]:
# --- Look: one customer, six orders, three of them over $200 ---
c0077 = window[window["customer_id"] == "C0077"]
print(f"orders over $200: {(c0077['total'] > 200).sum()}, total spend: ${c0077['total'].sum():.2f}")
c0077

## 1a. Build the per-customer table (6 points)

Because the comparison is between customers, the table must have one row for
every eligible customer. A customer in the pilot who placed no online order in
the window has no row in `window`. That customer must still appear in the table,
with zeros. You built the same kind of table for order counts in Lab 7 Step 1.

Build it in four steps:

| Step | Object | How |
|---|---|---|
| 1 | `eligible` | the snapshot rows with a `pilot` value, columns `customer_id` and `pilot`: `snap.loc[snap["pilot"].notna(), ["customer_id", "pilot"]]` |
| 2 | `qualifying` | the orders in `window` over \$200, counted per customer with `.groupby("customer_id").size()`, then `.rename("n_qualifying").reset_index()` |
| 3 | `spend` | `window.groupby("customer_id")["total"].sum()`, then `.rename("spend").reset_index()` |
| 4 | `per_customer` | `eligible` with `qualifying` and `spend` attached by two merges on `customer_id` with `how="left"`, then `.fillna(0)` on each of the two new columns |

- The filter in step 2 is a condition in square brackets, as in Chapter 3,
  section "Filtering rows": `window[window["total"] > 200]`.
- `.size()`, `.rename()` and `.reset_index()` are in Chapter 7, section "The
  pilot".
- The left merge and `.fillna(0)` are in Lab 7 Step 1. Each new column is
  replaced by its filled version: `per_customer["spend"] = per_customer["spend"].fillna(0)`.

In [ ]:
# TODO (1a): one row per eligible customer, zeros included
eligible = REPLACE_THIS        # customer_id and pilot, for customers with a pilot value
qualifying = REPLACE_THIS      # customer_id and n_qualifying, for customers with an order over $200
spend = REPLACE_THIS           # customer_id and spend, for customers with any online order

per_customer = REPLACE_THIS    # two left merges onto eligible
per_customer["n_qualifying"] = REPLACE_THIS    # missing counts become 0
per_customer["spend"] = REPLACE_THIS           # missing spend becomes 0

per_customer.head()

## Look at the shapes and the zeros

Because `qualifying` and `spend` contain only customers who ordered, they have
fewer rows than `eligible`. With a left merge, every row of `eligible` is kept.
`per_customer` should therefore have exactly as many rows as `eligible`. If it
has fewer, the customers who placed no order were dropped in a merge. The last
line prints C0077's row, which should show 3 and 1143.21.

In [ ]:
# --- Look: fewer rows before the merges, one row per eligible customer after ---
print("eligible:    ", eligible.shape)
print("qualifying:  ", qualifying.shape)
print("spend:       ", spend.shape)
print("per_customer:", per_customer.shape)
print((per_customer["spend"] == 0).sum(), "eligible customers placed no online order in the window")
per_customer[per_customer["customer_id"] == "C0077"]

In [ ]:
# --- Provided check: one row per eligible customer ---
assert len(per_customer) == len(eligible), "per_customer lost rows: use how=\"left\" in both merges"
print("check passes")

## Look at the two group means

In 1b to 1e you will compare these means. The row labelled 1.0 is the pilot
group and the row labelled 0.0 is the control group. For example, pilot
customers placed 0.309 qualifying orders on average.

In [ ]:
# --- Look: the mean of each outcome in each group ---
per_customer.groupby("pilot").agg(
    customers=("customer_id", "count"),
    n_qualifying=("n_qualifying", "mean"),
    spend=("spend", "mean"),
).round(3)

## 1b. Compute the difference and the Welch test (2 points)

Start with qualifying orders. `a` and `b` hold the two groups' values and are
provided in the cell.

- The difference in means is the pilot mean minus the control mean, as in
  Chapter 7, section "Testing the pilot".
- Welch's t-test is `stats.ttest_ind(a, b, equal_var=False)`, from the same
  section. It returns the t statistic and the p-value. You used it in Lab 7
  Step 4.

In [ ]:
# TODO (1b): the difference and the Welch t-test, qualifying orders
a = per_customer.loc[per_customer["pilot"] == 1, "n_qualifying"]    # pilot customers
b = per_customer.loc[per_customer["pilot"] == 0, "n_qualifying"]    # control customers

observed = REPLACE_THIS              # mean of a minus mean of b
t_stat, p_welch = REPLACE_THIS       # Welch's t-test

print(f"difference {observed:.3f} qualifying orders per customer")
print(f"Welch t = {t_stat:.2f}, p-value = {p_welch:.4f}")

## Look at one pass of the permutation loop

In the permutation test, one step is repeated 10,000 times. In the cell below,
that step is run once.

- `values` holds each customer's qualifying-order count, as a NumPy array.
- `is_pilot` holds one True or False per customer.
- After `rng.permutation(is_pilot)`, the same number of Trues is spread over
  different customers.
- The difference computed with the shuffled labels is one draw from a world in
  which the pilot had no effect.

In the output, the fifth customer is in the pilot group before the shuffle and in
the control group after it. With this one shuffle, the difference is −0.147.

In [ ]:
# --- Look: one shuffle of the group labels ---
values = per_customer["n_qualifying"].values       # one count per customer
is_pilot = (per_customer["pilot"] == 1).values      # True for pilot customers
rng = np.random.default_rng(0)
labels = rng.permutation(is_pilot)                  # one shuffle

print("actual labels:  ", is_pilot[:8])
print("shuffled labels:", labels[:8])
print(f"difference with the shuffled labels: {values[labels].mean() - values[~labels].mean():.3f}")

## 1c. Run the permutation test (3 points)

Repeat the shuffle 10,000 times and collect the differences. The p-value is the
share of shuffled differences that are at least as large as the observed
difference in absolute value. The loop and the p-value line are in Chapter 7,
section "Testing the pilot". You used the same code on order counts in Lab 7
Step 3.

In the first line, the generator is reset to seed 0. Your p-value will then
match everyone else's.

In [ ]:
# TODO (1c): the permutation test, qualifying orders
rng = np.random.default_rng(0)       # fixed seed
shuffled = []
for _ in range(10_000):
    REPLACE_THIS                     # two lines: shuffle the labels, append one difference
shuffled = np.array(shuffled)

p_perm = REPLACE_THIS                # share of |shuffled| at least |observed|
print(f"permutation p-value {p_perm:.4f}")

## 1d. Compute the bootstrap interval (2 points)

Resample each group with replacement, take the difference of the resampled
means, and repeat 10,000 times. The 2.5th and 97.5th percentiles of those
differences are the 95% interval. Chapter 7, section "The size of the effect",
has both lines: a list comprehension with `rng.choice`, then `np.percentile`.
You used them in Lab 7 Step 5.

In [ ]:
# TODO (1d): the 95% bootstrap interval, qualifying orders
rng = np.random.default_rng(0)       # fixed seed
boot = REPLACE_THIS                  # 10,000 resampled differences in means
lo, hi = REPLACE_THIS                # the 2.5th and 97.5th percentiles

print(f"difference {observed:.3f}, 95% CI [{lo:.3f}, {hi:.3f}]")

## 1e. Repeat the three tests for spend (3 points)

Run 1b, 1c and 1d again on `spend`. In the four provided lines, `a`, `b`,
`values` and `is_pilot` are redefined for spend. Below them, copy your code from
the three cells, in order, including each `rng = np.random.default_rng(0)` line.
Leave out the `a = ...` and `b = ...` lines of 1b. If those lines are copied, the
spend values are replaced with qualifying-order counts, and the results of 1b are
printed again.

Print the difference, the Welch p-value, the permutation p-value and the
interval.

In [ ]:
# TODO (1e): the same three steps for spend
a = per_customer.loc[per_customer["pilot"] == 1, "spend"]
b = per_customer.loc[per_customer["pilot"] == 0, "spend"]
values = per_customer["spend"].values
is_pilot = (per_customer["pilot"] == 1).values

REPLACE_THIS        # your code from 1b, 1c and 1d, in that order

## 1f. Interpretation of the two outcomes (9 points)

Apply one rule to each outcome. A difference is **distinguishable from chance at
the 5% level** when both p-values are below 0.05 and the 95% bootstrap interval
excludes zero. The same 0.05 convention is used in Chapter 7, section "Four rules
for reading p-values" (rule 3).

For each of the two outcomes, write:

- the difference in means, the Welch p-value and the permutation p-value;
- whether the 95% interval includes zero;
- one sentence of conclusion: the difference is, or is not, distinguishable from
  chance at the 5% level.

---

*Replace this line with your answer. This cell is graded.*

---

---

# Problem 2: Drivers of the discount rate (25 points)

`avg_discount_pct` in the snapshot is each customer's average discount rate over
twelve months, as a proportion: 0.061 is a 6.1% discount. `pct_online` is the
share of the customer's orders placed online, from 0 to 1. In this problem you
will model the discount rate with Chapter 8's multiple regression:

`avg_discount_pct ~ pct_online + orders_12m + C(business_type)`

## Look at the raw discount by business type

Before any model, run the cell below. The table lists each business type's mean
discount rate and mean online share. Schools have the highest mean discount,
0.068, and the lowest online share, 0.273. Offices have the lowest discount,
0.056, and the highest online share, 0.624. You will return to these two rows
in 2c.

In [ ]:
# --- Look: discount and online share by business type, before any adjustment ---
snap.groupby("business_type").agg(
    customers=("customer_id", "count"),
    discount=("avg_discount_pct", "mean"),
    online_share=("pct_online", "mean"),
).round(3)

## 2a. Fit the discount model (4 points)

Fit the model with `smf.ols(formula, data=snap).fit()` and call it `model`. The
formula is the one above. Both calls are in Chapter 8, sections "Regression
with one predictor" and "Regression with multiple predictors", and you used them
in Lab 8 Step 1. The cell prints the R² from `model.rsquared` and the coefficient table
from `model.summary().tables[1]`.

In [ ]:
# TODO (2a): fit the discount model
model = REPLACE_THIS

print(f"R² {model.rsquared:.3f}")
model.summary().tables[1]

## Look at one coefficient and at all of them

`model.params` is a Series of coefficients indexed by predictor name (Chapter 8,
section "Reading the regression table"). With one name in square brackets, a
single number is returned. Printed on its own, the Series lists every
coefficient with its name. The first line, about −0.0374, is the `pct_online`
row of the Series.

In [ ]:
# --- Look: one coefficient is a number; params is a Series ---
print(model.params["pct_online"])
model.params.round(4)

## 2b. Interpretation of the channel coefficient (4 points)

In two sentences:

- Convert the `pct_online` coefficient to a change of 10 percentage points in
  online share. Divide the coefficient by ten (Chapter 8, section "Regression
  with multiple predictors"), then multiply by 100, because the discount rate is
  a proportion. Name the two predictors held constant.
- State the coefficient's p-value and whether it is below 0.05.

---

*Replace this line with your answer. This cell is graded.*

---

## 2c. Business-type rows (3 points)

Apply one rule: a business type differs from the reference category only if its
p-value is below 0.05 (Chapter 8, sections "Reading the regression table" and
"Regression with multiple predictors").

In two sentences:

- Name the reference category.
- State the smallest p-value among the five business-type rows, and whether any
  business type differs from the reference.

---

*Replace this line with your answer. This cell is graded.*

---

## 2d. Compare the RMSE with the baseline (4 points)

Compute the model's root mean squared error and the error of the baseline that
predicts the mean discount for every customer. Both lines are in Chapter 8,
section "Evaluation: better than what, by how much?", and you used them in Lab 8
Step 4:

- the model: `np.sqrt((model.resid ** 2).mean())`;
- the baseline: the standard deviation of the outcome with `ddof=0`.

In [ ]:
# TODO (2d): the model's RMSE against the predict-the-mean baseline
rmse_model = REPLACE_THIS
rmse_baseline = REPLACE_THIS

print(f"model RMSE      {rmse_model:.4f}")
print(f"baseline RMSE   {rmse_baseline:.4f}")
print(f"error reduction {1 - rmse_model / rmse_baseline:.1%}")

## 2e. Drop the largest residual and refit (4 points)

You will repeat Chapter 8's stability check, from the section "The influential
customer". You used the same three steps in Lab 8 Step 5:

1. Find the customer with the largest residual in absolute value:
   `model.resid.abs().idxmax()` returns that row's index label.
2. Refit the same formula on `snap.drop(index=worst)`.
3. Compute the percentage change in each coefficient:
   `(model_trim.params - model.params) / model.params * 100`.

Because the two `params` Series share their index, every coefficient is
compared at once in the subtraction.

In [ ]:
# TODO (2e): drop the worst-fitting customer, refit, compare
worst = REPLACE_THIS           # index label of the largest absolute residual
model_trim = REPLACE_THIS      # the same formula, fitted without that row
change = REPLACE_THIS          # percentage change in each coefficient

pd.DataFrame({"before": model.params.round(4),
              "after": model_trim.params.round(4),
              "pct_change": change.round(1)})

## Look at the dropped customer

The refit is based on one row fewer than the original model. The dropped
customer, C0027, is an office with one order in the year, no online orders and a
discount of 0.157, more than twice the mean.

In [ ]:
# --- Look: one row fewer, and the row that was dropped ---
print("snap:      ", snap.shape)
print("refit data:", snap.drop(index=worst).shape)
snap.loc[[worst], ["customer_id", "business_type", "orders_12m", "pct_online", "avg_discount_pct"]]

## 2f. Usefulness and stability of the model (6 points)

Answer three questions, each with its number:

| Question | Rule |
|---|---|
| Is the model better than the baseline? | Yes if the error reduction from 2d is above 0%. |
| Is the `pct_online` coefficient stable? | Yes if its percentage change in 2e is smaller than 10% in absolute value. |
| Does the coefficient show the effect of moving a customer online? | Chapter 8, section "Reading the regression table": the coefficients of a regression on observational data are read as associations. The most direct way to measure a causal effect is a randomized experiment, as in Chapter 7. |

Write one sentence per question.

---

*Replace this line with your answer. This cell is graded.*

---

---

# Problem 3: Churn list under next year's costs (25 points)

In Chapter 9 a retention offer cost \$60 and a lost customer cost \$600 in margin.
Next year's figures are different:

| Cost | Chapter 9 | Next year |
|---|---|---|
| one retention offer | \$60 | \$50 |
| one lost customer | \$600 | \$250 |

In this problem you will refit the Chapter 9 model and rebuild its cost analysis
with next year's figures.

## 3a. Split the data and fit the logistic pipeline (3 points)

- Use Chapter 9's split exactly:
  `train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)`. It
  returns `X_train, X_test, y_train, y_test`, in that order. With the same
  arguments you obtain the chapter's 610 training and 262 test customers.
- Make the pipeline `make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))`
  and call it `logit`. In the provided lines, the pipeline is fitted on the
  training set and its test accuracy is printed with `accuracy_score`.

Both steps are in Chapter 9, sections "The train/test discipline" and "Logistic
regression". You used them in Lab 9 Step 2.

In [ ]:
# TODO (3a): the Chapter 9 split and the logistic pipeline
X_train, X_test, y_train, y_test = REPLACE_THIS

logit = REPLACE_THIS           # the scaler and the logistic model, in one pipeline
logit.fit(X_train, y_train)

print(f"test accuracy {accuracy_score(y_test, logit.predict(X_test)):.3f}  (majority baseline 0.780)")

## Look at the split

The training set has 610 rows and the test set 262, each with the 20 feature
columns. With `stratify=y`, the churn rate is 0.220 in the training set and 0.221
in the test set.

In [ ]:
# --- Look: 610 training and 262 test customers, with the same churn rate ---
print(X_train.shape, X_test.shape)
print(f"churn rate: training {y_train.mean():.3f}, test {y_test.mean():.3f}")

## 3b. Compute the confusion matrix at 0.5 (3 points)

`logit.predict` returns 1 when a customer's churn probability is above 0.5.
Compute `confusion_matrix(y_test, logit.predict(X_test))` and call it `cm`. In
the provided lines, `cm` is wrapped in a labelled table, as in Chapter 9, section
"The confusion matrix". You used the same call in Lab 9 Step 3.

In [ ]:
# TODO (3b): the confusion matrix at the default threshold of 0.5
cm = REPLACE_THIS

pd.DataFrame(cm, index=["actually stayed", "actually churned"],
             columns=["predicted stay", "predicted churn"])

## Look at the probabilities and one threshold

- `predict_proba` returns a NumPy array with two columns per customer: the
  probability of staying, then the probability of churning.
- With `[:, 1]`, the churn column is kept for every customer. `proba` is that
  column, one number per test customer.
- `proba >= 0.2` holds one True or False per customer, and `.sum()` returns the
  number of Trues. The comparison at 0.2 is one pass of the loop in 3c.

All three lines are explained in Chapter 9, section "Moving the threshold". In
the output, the first test customer has a churn probability of 0.044, and 57
customers are flagged at 0.2.

In [ ]:
# --- Look: two columns, then one column, then one threshold ---
print(logit.predict_proba(X_test)[:3].round(3))     # three customers: P(stay), P(churn)
proba = logit.predict_proba(X_test)[:, 1]            # P(churn) for every test customer
print(proba[:3].round(3))

flagged = proba >= 0.2
print(flagged[:3], f"{flagged.sum()} customers flagged at 0.2")

## 3c. Build the cost table at four thresholds (6 points)

For each threshold in `[0.5, 0.35, 0.2, 0.1]`:

- flag the customers whose churn probability is at or above the threshold;
- count the churners caught, the false alarms and the churners missed;
- compute the error cost with next year's figures,
  `OFFER * (caught + false_alarms) + LOSS * missed`;
- append one row to `rows`.

Chapter 9, section "Moving the threshold", has this loop with \$60 and \$600.
You used it in Lab 9 Step 5. `OFFER` and `LOSS` are set in the cell.

In [ ]:
# TODO (3c): the cost table with next year's figures
OFFER = 50      # next year's cost of one retention offer
LOSS = 250      # next year's cost of one lost customer

rows = []
for threshold in [0.5, 0.35, 0.2, 0.1]:
    REPLACE_THIS          # flagged, tp, fp, fn and cost; then append one row

pd.DataFrame(rows, columns=["threshold", "caught", "false alarms", "missed", "error cost"])

## 3d. Derive the break-even threshold and compare with the baselines (4 points)

- The break-even threshold is the cost of an offer divided by the cost of a lost
  customer (Chapter 9, section "Moving the threshold"). Compute it from `OFFER`
  and `LOSS`.
- Write the two-argument `error_cost(flagged, labels)` from Chapter 10, section
  "Four models compared by cross-validation", with `OFFER` and `LOSS` as the two
  costs. You will use it again in Problem 4.
- Enter the four test-set costs: no offers (`LOSS` times the number of
  churners), an offer to every customer (`OFFER` times the number of customers),
  and the model at 0.5 and at the break-even threshold. Chapter 9, section
  "Evaluation against two baselines", has the same table.

In [ ]:
# TODO (3d): the break-even threshold, error_cost, and the two baselines
break_even = REPLACE_THIS

def error_cost(flagged, labels):
    """Total error cost: OFFER per offer, LOSS per churner without an offer."""
    REPLACE_THIS          # tp, fp and fn from flagged and labels; return the cost

print(f"break-even threshold {break_even:.2f}")
pd.Series({
    "no offers (every churner lost)": REPLACE_THIS,
    "offer to every customer": REPLACE_THIS,
    "model at threshold 0.5": REPLACE_THIS,
    "model at the break-even threshold": REPLACE_THIS,
}).to_frame("test-set error cost")

## 3e. Interpretation of the new costs (9 points)

Write one sentence for each item, with its numbers:

1. The break-even threshold, computed as the cost of an offer divided by the cost
   of a lost customer.
2. The model's test cost at the break-even threshold, and whether it is below
   both baselines.
3. The saving against an offer to every customer, in dollars and as a percentage
   of that baseline's cost, and whether it is larger than Chapter 9's \$1,680.
4. The number of test customers flagged at the break-even threshold, against 114
   at Chapter 9's threshold of 0.1, and the number of churners missed, against 12.
5. The assumption about retention offers on which every cost figure depends
   (Chapter 9, section "Evaluation against two baselines").

---

*Replace this line with your answer. This cell is graded.*

---

---

# Problem 4: Churn list without order dates (25 points)

Order dates may be missing from the new customer export. Without them,
`weeks_since_last_order` cannot be computed. Recency had by far the largest
importance in Chapter 10's forest. In this problem you will repeat Chapter 10's
model selection without recency, at next year's break-even threshold of 0.2, and
measure the cost of losing it.

## 4a. Drop recency from the features (2 points)

Make `Xn_train` and `Xn_test` from `X_train` and `X_test` with
`.drop(columns=["weeks_since_last_order"])`, as `snap.drop(columns=[...])` was
used in Chapter 9, section "The train/test discipline". Because the rows are
unchanged, the split is the same as in Problem 3.

In [ ]:
# TODO (4a): the feature tables without recency
Xn_train = REPLACE_THIS
Xn_test = REPLACE_THIS

## Look at the columns

Each feature table has the same rows as before and one column fewer, 19 feature
columns. Recency is the dropped column.

In [ ]:
# --- Look: the same rows, one column fewer ---
print("with recency:   ", X_train.shape, X_test.shape)
print("without recency:", Xn_train.shape, Xn_test.shape)

## 4b. Choose the tree depth by cross-validation (4 points)

For each depth from 1 to 8:

- create `DecisionTreeClassifier(max_depth=depth, random_state=0)`;
- compute its five-fold CV accuracy with `cross_val_score(model, Xn_train, y_train, cv=5)`;
- fit it on `Xn_train` and record its training accuracy with `.score`;
- append the depth, the training accuracy, the CV mean and the CV standard
  deviation.

Chapter 10, section "Cross-validation", has this loop. Choose the depth with the
highest CV mean accuracy, unless the second-highest is within one standard
deviation of it; in that case, treat the two as tied and choose the less flexible
(shallower) one, as in Chapter 10. Use the chosen depth in 4c.

In [ ]:
# TODO (4b): the depth sweep by cross-validation, without recency
rows = []
for depth in [1, 2, 3, 4, 5, 6, 7, 8]:
    REPLACE_THIS        # model, cv_scores, fit; then append one row

pd.DataFrame(rows, columns=["max_depth", "train", "CV mean", "CV std"]).round(3)

## Look at the depth-1 tree

A depth-1 tree consists of a single question. `export_text` returns the
question as text, with the counts of customers who stayed and churned in each
leaf (Chapter 10, section "The fitted tree"). Because both leaves have the class
0, this tree predicts "stay" for every customer. Its churn probabilities still
differ by leaf. The first leaf holds the customers with twelve-month revenue of
\$2,758.29 or less. In that leaf, 94 of 284 customers churned, a share of
0.331, above the break-even threshold of 0.2.

In [ ]:
# --- Look: the single question of a depth-1 tree without recency ---
tree_1 = DecisionTreeClassifier(max_depth=1, random_state=0)
tree_1.fit(Xn_train, y_train)
print(export_text(tree_1, feature_names=list(Xn_train.columns), show_weights=True))

## Look at cross-validated probabilities

`cross_val_predict` with `method="predict_proba"` returns, for every training
customer, a churn probability from a model that was not fitted on that customer
(Chapter 10, section "Four models compared by cross-validation"). The first
three probabilities of the logistic model are printed beside the three actual
labels. The first customer stayed and has a probability of 0.368, above 0.2. At
the break-even threshold, that customer would receive an offer, and the offer
would be a false alarm. In 4c the same call is made once per model.

In [ ]:
# --- Look: cross-validated churn probabilities, logistic model, without recency ---
proba_cv = cross_val_predict(logit, Xn_train, y_train, cv=5,
                             method="predict_proba")[:, 1]
print(proba_cv[:3].round(3), y_train.values[:3])

## 4c. Compare three models by cross-validation (6 points)

Compare three models on `Xn_train`:

- logistic regression, the pipeline from 3a;
- the tree at the depth you chose in 4b, with `random_state=0`;
- the default forest, `RandomForestClassifier(n_estimators=400, random_state=0)`.

For each model, record the CV accuracy (mean ± standard deviation) and the CV
error cost at the break-even threshold of 0.2, using your `error_cost` from 3d.
The two baselines on the training set are provided as the first two rows.
Chapter 10, section "Four models compared by cross-validation", has the
dictionary of models and the loop, and you used `cross_val_predict` in Lab 10
Step 4.

Select the model with the lowest CV error cost. You will use it in 4e.

In [ ]:
# TODO (4c): three models and two baselines, by cross-validation
models = {
    "logistic regression": REPLACE_THIS,
    "tree": REPLACE_THIS,                 # the depth you chose in 4b
    "random forest": REPLACE_THIS,
}

rows = [["no offers (baseline)", f"{1 - y_train.mean():.3f}", f"${LOSS * y_train.sum():,}"],
        ["offer to every customer (baseline)", f"{y_train.mean():.3f}", f"${OFFER * len(y_train):,}"]]
for name, model in models.items():
    REPLACE_THIS        # cv_scores, proba_cv; then append one row

pd.DataFrame(rows, columns=["policy", "CV accuracy", "CV error cost at 0.2"])

## 4d. Measure the cost of losing recency (2 points)

Compute the CV error cost at 0.2 of the logistic pipeline on the full `X_train`,
with recency. You will need one `cross_val_predict` call, as in 4c, and one
`error_cost` call. Compare the result with the logistic row of 4c.

In [ ]:
# TODO (4d): the logistic model's CV error cost with recency
proba_cv_full = REPLACE_THIS       # CV churn probabilities on X_train, all 20 features
cost_full = REPLACE_THIS           # error cost at 0.2

print(f"CV error cost at 0.2 with recency: ${cost_full:,}")

## 4e. Report the selected model on the test set (2 points)

Fit the model with the lowest CV error cost in 4c on `Xn_train` and `y_train`,
then report its test accuracy and its test error cost at 0.2. Use the test set
once, after the selection. The test-set baselines are the ones from 3d. Chapter
10, section "Evaluation against two baselines", has the same steps.

In [ ]:
# TODO (4e): the selected model, used once on the test set
selected = REPLACE_THIS            # a new, unfitted copy of the model you selected
selected.fit(Xn_train, y_train)
proba_test = REPLACE_THIS          # P(churn) for every test customer

print(f"test accuracy {selected.score(Xn_test, y_test):.3f}")
print(f"test error cost at 0.2 ${error_cost(proba_test >= 0.2, y_test):,}")

## 4f. Interpretation and recommendation (9 points)

Write one sentence for each item, with its numbers:

1. Accuracy. A model is more accurate than the majority baseline of 0.780 only if
   its CV mean exceeds 0.780 by more than its CV standard deviation (Chapter 10,
   section "Cross-validation"). State whether any of the three models passes.
2. The selected model's CV error cost, and whether it is below both CV
   baselines.
3. The selected model's test error cost, and whether it is below both test-set
   baselines from 3d.
4. The value of recency: the logistic model's CV error cost without recency (4c)
   minus its CV error cost with recency (4d), in total and per training customer
   (divide by 610).
5. The recommendation, by rule: if the CV error cost is lower with recency, order
   dates should be kept in the export.

---

*Replace this line with your answer. This cell is graded.*

---

---

## Submission checklist

1. **Restart and run all.** Confirm that no error is raised in any cell.
2. Check that every cell marked **graded** contains your answer. The placeholder
   text should be gone.
3. Download as `.ipynb` and upload it to the Assignment 2 dropbox.
4. If you used an AI assistant, add a line below naming the tool and what it
   helped with.

*AI note (if applicable):*